# Task 2: Original LCS System on Raw Dataset

## Imports

In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer, RobustScaler, KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from skeLCS import eLCS
import time

## Loading raw dataset

In [4]:
#loading the raw dataset for the baseline of LCS
df_raw_lcs = pd.read_csv('MattyLuriz_creditcard.csv')
df_raw_lcs.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Minimal Processing: exact-count stratified subsample

In [5]:
#LCS compatibility requires minimal processing
# Rather than subsampling via train_test_split's stratify argument, the fraud
# rate in the raw data is used to compute exact per-class sample sizes, then
# each class is sampled independently. This keeps the subsample's fraud rate
# tied explicitly to the raw dataset's rate rather than relying on stratify
# to preserve it implicitly.
n_total = 20000
fraud_rate = df_raw_lcs['Class'].mean()
n_fraud = round(n_total * fraud_rate)
n_legit = n_total - n_fraud

fraud_sample = df_raw_lcs[df_raw_lcs['Class'] == 1].sample(n=n_fraud, random_state=42)
legit_sample = df_raw_lcs[df_raw_lcs['Class'] == 0].sample(n=n_legit, random_state=42)

df_sample = pd.concat([fraud_sample, legit_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Raw fraud rate: {fraud_rate:.4%}")
print(f"Subsample size: {df_sample.shape[0]} rows ({n_fraud} fraud, {n_legit} legitimate)")
print(f"Subsample fraud rate: {df_sample['Class'].mean():.4%}")

Raw fraud rate: 0.1727%
Subsample size: 20000 rows (35 fraud, 19965 legitimate)
Subsample fraud rate: 0.1750%


In [6]:
X = df_sample.drop(columns=['Class']).values
y = df_sample['Class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Training set: {X_train.shape}, Fraud cases: {y_train.sum()}")
print(f"Test set: {X_test.shape}, Fraud cases: {y_test.sum()}")

Training set: (16000, 30), Fraud cases: 28
Test set: (4000, 30), Fraud cases: 7


## Run the original, unmodified eLCS baseline

In [8]:
model = eLCS(learning_iterations=10000, random_state=42)

start = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start

preds = model.predict(X_test)

print(f"Training time: {elapsed:.2f} seconds")
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, preds, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, preds, zero_division=0):.4f}")

Training time: 31.98 seconds
Accuracy: 0.9985
Balanced Accuracy: 0.5714
Precision: 1.0000
Recall: 0.1429
F1 Score: 0.2500


### Task 2 Results: Original LCS System on Raw Dataset

An unmodified eLCS baseline was run on the unpreprocessed, uncleaned dataset. Since applying LCS rule evolution to 284,807 rows was computationally infeasible, a subsample was drawn of 20,000 rows (large enough to accommodate the original dataset's fraud rate of 0.1727% without dividing the rate at random) and split into training (16,000 rows) and test (4,000 rows) sets (with 28 and 7 frauds respectively). The subsample (including the target variable's position) was then reshuffled and all features were rearranged with the target in the first position to form an array that could be fed into the eLCS classifier without any other cleaning, scaling, or transforming (as this is to be used for the unpreprocessed baseline).

The model was ran with eLCS's unmodified, default parameters(learning_iterations = 10000, N = 1000, p_spec = 0.5, nu = 5, chi = 0.8, mu = 0.04, theta_GA = 25), training being completed at 31.98 seconds

**Baseline results**:
|  Metric  |  Value  |  
|----------|---------|  
| Accuracy | 0.9985  |  
| Balanced Accuracy | 0.5714 |  
|Precision | 1.0000  |  
| Recall   | 0.1429  |  
| F1-score | 0.2500  |  

The headline accuracy value provides no useful indication of performance here: in a problem space where less than 0.2% of the cases represent fraud, a classifier assigning "legitimate" to all cases would produce nearly the same score. The balanced accuracy score of 0.5714 gives a better, though still not exceptional reflection (just above 0.5, being the chance baseline). Most revealing are the precision/recall metrics, which tell the true story of the system: its constrained and highly selective rule population meant that whenever it predicted a transaction to be fraud, it was 100% correct in its claim (precision = 1.0). 

The price paid for this precision was its failure to generalize, capturing only 1 out of the 7 fraud cases (recall = 0.1429), leaving the remaining fraud cases in the test set undetected. 

With only 28 training fraud examples, there was not enough minority-class signal for eLCS to build a rule population generalized to cover most of the cases, even though it found one reliable signature of a fraud case. This low-recall, near-chance baseline provides a target that the pre-processing and class-imbalance-mitigation work of Tasks 3 and 4 intends to overcome.